In [1]:
# -*- coding: utf-8 -*-
"""
us_export_data_collect.py
--------------------------
미국 Census API → HS 코드별 수출(Export) 월별 데이터 수집 후 MySQL 저장

폴더 구조:
    stock_forecast/
    ├── DATA/
    │   ├── config.py                  ← DB 정보·공통 설정
    │   └── us_top_export_hs_code.py   ← HS 코드 리스트
    └── US_Market/collect/us_trade_export_data/
        └── us_export_data_collect.py  ← 이 파일

최초 1회 패키지 설치:
    pip install aiohttp nest_asyncio tqdm pymysql sqlalchemy
"""

# ── 표준 라이브러리 ──────────────────────────────────────────────────────────
import asyncio
import sys
import os
import time
from pathlib import Path
from typing import List, Optional

# ── 서드파티 ─────────────────────────────────────────────────────────────────
import nest_asyncio          # Jupyter 이벤트 루프 충돌 방지 (핵심 수정)
nest_asyncio.apply()         # 반드시 다른 import 보다 먼저 적용

import aiohttp
import pandas as pd
from sqlalchemy import text
from tqdm import tqdm

# ══════════════════════════════════════════════════════════════════════════════
# 1. 범용 경로 설정 — DATA 폴더 자동 탐색
# ══════════════════════════════════════════════════════════════════════════════
def setup_universal_paths() -> dict:
    current = Path.cwd()
    for parent in [current, *current.parents]:
        data_folder = parent / "DATA"
        if data_folder.exists():
            for p in (str(parent), str(data_folder)):
                if p not in sys.path:
                    sys.path.insert(0, p)
            print("=" * 70)
            print("경로 설정 완료")
            print("=" * 70)
            print(f"  프로젝트 루트 : {parent}")
            print(f"  DATA 폴더     : {data_folder}")
            print(f"  현재 위치     : {current}")
            print(f"  운영체제      : {os.name}")
            print("=" * 70 + "\n")
            return {"project_root": parent, "data_folder": data_folder, "current": current}
    raise FileNotFoundError(
        f"DATA 폴더를 찾을 수 없습니다.\n현재 위치: {current}"
    )

try:
    paths = setup_universal_paths()
except FileNotFoundError as e:
    print(e)
    sys.exit(1)

# ══════════════════════════════════════════════════════════════════════════════
# 2. 프로젝트 내부 모듈 import
#    config.py 에 실제로 존재하는 이름만 사용
# ══════════════════════════════════════════════════════════════════════════════
from config import (
    get_db_info,        # DB 접속 정보 dict 반환
    get_engine,         # SQLAlchemy engine 생성
    log,                # [TAG] message 출력
    BATCH_SIZE_DEFAULT, # 배치 기본값 (=20)
    START_DATE_MONTH,   # 수집 기본 시작일
)
from us_top_export_hs_code import US_TOP_EXPORT_HS_CODES

# Census API 키 — config.py 에 없으므로 여기서 직접 정의
API_KEY = "bf388499b71a365d725e1c888201736f7409d7e4"

# ══════════════════════════════════════════════════════════════════════════════
# 3. 수집 파라미터
# ══════════════════════════════════════════════════════════════════════════════
COLLECT_START  = "2016-01"

# 현재 달 기준 2개월 전 (Census API 최신 데이터는 미확정이므로 제외)
COLLECT_END    = (pd.Timestamp.today() - pd.DateOffset(months=2)).strftime("%Y-%m")

TABLE_NAME     = "us_export_data"

MAX_CONCURRENT = 25      # 동시 요청 수 — Census API 안전 한도
RETRY_COUNT    = 3       # 실패 시 재시도 횟수
RETRY_DELAY    = 2.0     # 재시도 대기 (초)
DB_CHUNK_SIZE  = 1_000   # DB INSERT 청크 크기

# ══════════════════════════════════════════════════════════════════════════════
# 4. DB DDL / SQL
# ══════════════════════════════════════════════════════════════════════════════
DDL = f"""
CREATE TABLE IF NOT EXISTS `{TABLE_NAME}` (
    `id`         BIGINT      NOT NULL AUTO_INCREMENT,
    `hs_code`    VARCHAR(10) NOT NULL  COMMENT 'HS 6자리 코드',
    `date`       DATE        NOT NULL  COMMENT '해당 월의 말일',
    `year`       CHAR(4)     NOT NULL,
    `month`      CHAR(2)     NOT NULL,
    `exp_dlr`    BIGINT               COMMENT '수출금액 (USD)',
    `created_at` DATETIME    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    `updated_at` DATETIME    NOT NULL DEFAULT CURRENT_TIMESTAMP
                                       ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (`id`),
    UNIQUE KEY `uq_hs_date` (`hs_code`, `date`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
  COMMENT='미국 HS 코드별 월별 수출 데이터';
"""

INSERT_SQL = f"""
INSERT IGNORE INTO `{TABLE_NAME}`
    (hs_code, date, year, month, exp_dlr)
VALUES
    (:hs_code, :date, :year, :month, :exp_dlr)
"""

# ══════════════════════════════════════════════════════════════════════════════
# 5. DB 유틸 함수
# ══════════════════════════════════════════════════════════════════════════════
def ensure_table(engine) -> None:
    """테이블이 없으면 자동 생성"""
    with engine.connect() as conn:
        conn.execute(text(DDL))
        conn.commit()
    log("DB", f"테이블 '{TABLE_NAME}' 준비 완료")


def load_existing_keys(engine) -> set:
    """이미 저장된 (hs_code, 'yyyy-mm') 조합 → API 호출 자체를 건너뜀"""
    with engine.connect() as conn:
        rows = conn.execute(
            text(f"SELECT hs_code, DATE_FORMAT(date, '%Y-%m') FROM `{TABLE_NAME}`")
        ).fetchall()
    existing = {(r[0], r[1]) for r in rows}
    log("DB", f"기존 저장 건수: {len(existing):,} 개")
    return existing


def save_to_db(records: List[dict], engine) -> tuple:
    """INSERT IGNORE 방식 저장. 반환값: (신규저장, 중복스킵)"""
    if not records:
        return 0, 0

    rows = []
    for r in records:
        val = r.get("exp_dlr")
        try:
            val_num = int(val) if val not in (None, "None", "") else None
            if val_num is not None and val_num > 1_000_000_000_000_000_000:
                val_num = None
        except (ValueError, TypeError):
            val_num = None

        date_obj = (
            pd.Timestamp(f"{r['year']}-{r['month']}-01") + pd.offsets.MonthEnd(0)
        ).date()

        rows.append({
            "hs_code": r["hs_code"],
            "date":    date_obj,
            "year":    r["year"],
            "month":   r["month"],
            "exp_dlr": val_num,
        })

    inserted = 0
    with engine.connect() as conn:
        for i in range(0, len(rows), DB_CHUNK_SIZE):
            chunk = rows[i : i + DB_CHUNK_SIZE]
            result = conn.execute(text(INSERT_SQL), chunk)
            inserted += result.rowcount
        conn.commit()

    return inserted, len(rows) - inserted


# ══════════════════════════════════════════════════════════════════════════════
# 6. 비동기 API 요청
# ══════════════════════════════════════════════════════════════════════════════
BASE_URL = "https://api.census.gov/data/timeseries/intltrade/exports/hs"


async def fetch_one(
    session: aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    hs: str,
    year: str,
    month: str,
) -> Optional[dict]:
    url = (
        f"{BASE_URL}?get=ALL_VAL_MO"
        f"&key={API_KEY}"
        f"&YEAR={year}&MONTH={month}&E_COMMODITY={hs}"
    )
    for attempt in range(1, RETRY_COUNT + 1):
        async with semaphore:
            try:
                async with session.get(
                    url, timeout=aiohttp.ClientTimeout(total=30)
                ) as resp:
                    if resp.status == 200:
                        data = await resp.json(content_type=None)
                        if len(data) > 1:
                            return {
                                "hs_code": hs,
                                "year":    year,
                                "month":   month,
                                "exp_dlr": data[1][0],
                            }
                        return None
                    elif resp.status == 429:
                        await asyncio.sleep(RETRY_DELAY * attempt)
                    else:
                        return None
            except Exception:
                if attempt < RETRY_COUNT:
                    await asyncio.sleep(RETRY_DELAY)
    return None


async def collect_all(tasks: List[tuple]) -> List[dict]:
    """전체 태스크를 MAX_CONCURRENT 동시 요청으로 처리"""
    FLUSH      = 500
    semaphore  = asyncio.Semaphore(MAX_CONCURRENT)
    results    = []

    connector = aiohttp.TCPConnector(limit=MAX_CONCURRENT, ssl=False)
    async with aiohttp.ClientSession(connector=connector) as session:
        with tqdm(
            total=len(tasks), desc="수출 데이터 수집",
            unit="req", ncols=90, file=sys.stdout
        ) as pbar:
            for i in range(0, len(tasks), FLUSH):
                batch  = tasks[i : i + FLUSH]
                coros  = [fetch_one(session, semaphore, hs, yr, mo)
                          for hs, yr, mo in batch]
                batch_results = await asyncio.gather(*coros)
                results.extend(r for r in batch_results if r is not None)
                pbar.update(len(batch))

    return results


# ══════════════════════════════════════════════════════════════════════════════
# 7. 메인
# ══════════════════════════════════════════════════════════════════════════════
def main():
    t0 = time.time()
    log("START", f"미국 수출 데이터 수집 시작 | {COLLECT_START} ~ {COLLECT_END}")
    log("INFO",  f"수집 대상 HS 코드: {len(US_TOP_EXPORT_HS_CODES):,} 개")

    # DB 준비
    db_info = get_db_info()
    engine  = get_engine(db_info)
    ensure_table(engine)

    # 이미 저장된 키 로드 → 중복 API 호출 방지
    existing   = load_existing_keys(engine)
    date_range = pd.date_range(start=COLLECT_START, end=COLLECT_END, freq="MS")

    tasks = [
        (hs, dt.strftime("%Y"), dt.strftime("%m"))
        for hs in US_TOP_EXPORT_HS_CODES
        for dt in date_range
        if (hs, dt.strftime("%Y-%m")) not in existing
    ]

    total_possible = len(US_TOP_EXPORT_HS_CODES) * len(date_range)
    log("INFO", (
        f"전체 가능 조합: {total_possible:,} 개 | "
        f"기존 보유: {len(existing):,} 개 | "
        f"신규 수집 대상: {len(tasks):,} 개"
    ))

    if not tasks:
        log("DONE", "신규 수집할 데이터가 없습니다. 종료.")
        return

    log("FETCH", f"동시 요청 수: {MAX_CONCURRENT}")

    # ── asyncio 실행 ────────────────────────────────────────────────────────
    # nest_asyncio.apply() 가 상단에서 이미 적용됐으므로
    # Jupyter / .py 스크립트 어느 환경에서도 asyncio.run() 정상 동작
    records = asyncio.run(collect_all(tasks))
    log("FETCH", f"API 응답 성공: {len(records):,} 건")

    # DB 저장
    inserted, skipped = save_to_db(records, engine)

    elapsed = time.time() - t0
    log("DONE", (
        f"신규 저장: {inserted:,} 건 | "
        f"중복 스킵: {skipped:,} 건 | "
        f"소요시간: {elapsed:.1f} 초 ({elapsed / 60:.1f} 분)"
    ))


# ── 스크립트 직접 실행 시 ──────────────────────────────────────────────────
if __name__ == "__main__":
    main()


경로 설정 완료
  프로젝트 루트 : C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast
  DATA 폴더     : C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA
  현재 위치     : C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\US_Market\collect\us_trade_export_data
  운영체제      : nt



C:\Users\Hoyoung_Park\PyCharmMiscProject\.venv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[START] 미국 수출 데이터 수집 시작 | 2016-01 ~ 2026-03
[INFO] 수집 대상 HS 코드: 500 개
[DB] 테이블 'us_export_data' 준비 완료
[DB] 기존 저장 건수: 58,594 개
[INFO] 전체 가능 조합: 61,500 개 | 기존 보유: 58,594 개 | 신규 수집 대상: 2,906 개
[FETCH] 동시 요청 수: 25
수출 데이터 수집: 100%|██████████████████████████████| 2906/2906 [00:57<00:00, 50.22req/s]
[FETCH] API 응답 성공: 500 건
[DONE] 신규 저장: 500 건 | 중복 스킵: 0 건 | 소요시간: 61.4 초 (1.0 분)
